# ContractRisk M3: desarrollo v2

Comparación de recuperación: denso, híbrido con filtro de ID, e híbrido con reranking. Mismo prompt v2 y generador en todas las variantes. Cuatro preguntas de desarrollo y un caso adicional de contexto vacío, sin gold final. No es aún el harness final, el agente o RAGAS. Usa Colab T4 y un entorno nuevo. Sube contractrisk_m3_desarrollo_v2_codigo.zip. La solución SQLite es automática; no copies las celdas de reparación anteriores.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys, subprocess
uploaded = files.upload()
name = 'contractrisk_m3_desarrollo_v2_codigo.zip'
assert name in uploaded, f'Sube {name}'
root = Path('/content/contractrisk_m3_desarrollo_v2')
assert not root.exists(), 'Usa un entorno nuevo; no mezcles corridas.'
with zipfile.ZipFile(name) as z:
    for member in z.namelist():
        assert (root / member).resolve().is_relative_to(root.resolve()), 'Ruta insegura'
    z.extractall(root)
os.chdir(root)
print(root)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r','m3/requirements-development-v2.txt'],check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s','m3/tests','-v'],check=True)
subprocess.run([sys.executable,'-c',"import sys;sys.path.insert(0,'m3');from sqlite_support import ensure_sqlite;print(ensure_sqlite());import torch;assert torch.cuda.is_available(),'Activa T4';print(torch.cuda.get_device_name(0))"],check=True)

In [ ]:
command = [sys.executable,'-u','m3/run_development_v2.py','--output','desarrollo_v2_resultados']
with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
    for line in process.stdout:
        print(line,end='',flush=True)
    returncode = process.wait()
if returncode:
    print('La corrida falló. Ejecuta la siguiente celda para descargar el diagnóstico.')
else:
    print('Corrida terminada. Descarga los resultados para revisar calidad.')

In [ ]:
import shutil
resultados = Path('desarrollo_v2_resultados')
assert resultados.exists(), 'Si falló antes de crear resultados, envía el error completo de instalación/preflight.'
print((resultados/'metadata.json').read_text())
archive = shutil.make_archive('/content/contractrisk_m3_desarrollo_v2_resultados','zip',resultados)
files.download(archive)

Devuelve el ZIP y este notebook con sus salidas. Conserva la corrida anterior. No ajustes los prompts ni los datos para subir puntuaciones. Los 12 casos gold no se ejecutan en esta prueba.